# Work with Embeddings: Create, Upload, Compare

This notebook explores possibilities to profit from embeddings of a collection of sources. The collection is the digital collection of sources of the project [The School of Salamanca](https://salamanca.school/), and the notebook uses multilingual embeddings from OpenAI, Cohere and, eventually, Ollama.

It makes use of a vectore store API that has been developed for these experiments: the Vector database of the DH at Max Planck Society initiative (dhamps-vdb), developed by [Andreas Wagner](https://www.lhlt.mpg.de/wagner/en) of the [Max Planck Institute for Legal History and Legal Theory](https://www.lhlt.mpg.de/).

## 0. Preliminaries

In [31]:
# Get info about python version
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

/home/awagner/vcs/digicademy/svsal-bertopic/.venv/bin/python
3.9.20 (main, Oct 16 2024, 04:36:33) 
[Clang 18.1.8 ]
sys.version_info(major=3, minor=9, micro=20, releaselevel='final', serial=0)


## 1. Setup

### 1.1 Install libraries

Instead of using the below python/ipython commands, and in order to make the notebook more declarative/reproducible, we try to define the necessary libraries and environment in a `uv` *project*, i.e. in the [pyproject.toml file](./pyproject.toml) that controls how `uv` manages the `.venv` virtual environment.

According to the [uv documentation](https://docs.astral.sh/uv/concepts/projects/layout/#the-project-environment):

> To run a command in the project environment, use `uv run`. Alternatively the project environment can be activated as normal for a virtual environment.
>
> When `uv run` is invoked, it will create the project environment if it does not exist yet or ensure it is up-to-date if it exists. The project environment can also be explicitly created with `uv sync`.
>
> It is *not* recommended to modify the project environment manually, e.g., with `uv pip install`. For project dependencies, use `uv add` to add a package to the environment.

### 1.2 Load libraries

In [32]:
import os
import dotenv
import itertools
from typing import Any, Dict, List
from datetime import datetime
from abc import ABC, abstractmethod
import rich.progress
import rich.live
import rich.console

import polars as pl
import numpy as np
import pickle
import orjson

import asyncio
import nest_asyncio

import tiktoken
import transformers
import openai
import cohere

## Provider Configuration Section

In [33]:
# Input files
file_path_in_text = './in-data/all_paragraphs_20250411.csv'  # text and metadata

# Output files
file_path_out = "./out-data"
file_path_prefix = datetime.now().strftime("%Y-%m-%d") + "_"
file_path_out_docs_pickle         = file_path_out + "/" + file_path_prefix + "full_docs.pkl"
file_path_out_docs_parquet        = file_path_out + "/" + file_path_prefix + "full_docs.parquet"
file_path_out_docs_csv            = file_path_out + "/" + file_path_prefix + "docs.csv"
file_path_out_embeddings_pickle   = file_path_out + "/" + file_path_prefix + "embeddings.pkl"
file_path_out_embeddings_parquet  = file_path_out + "/" + file_path_prefix + "embeddings.parquet"
file_path_out_vdb_payloads_jsonl  = file_path_out + "/" + file_path_prefix + "embeddings.jsonl"
file_path_out_config_json         = file_path_out + "/" + file_path_prefix + "processing_metadata.json"

# General limits
max_documents = 5    # Set to -1 to process all documents
min_tokens = 10      # Minimum number of tokens for a paragraph to be considered

# Provider configurations
providers_config = {
    # OpenAI Configuration
    "openai": {
        "enabled": True,
        "name": "openai",
        "model": "text-embedding-3-small",
        "dimensions": 1536,
        "context_limit": 8191,
        "encoding": "cl100k_base",
        "embedding_type": "float", # alternative: base64
        "rate_limit_per_minute": 3000,
        "concurrent_requests": 30,
        "base_url": None  # Use default OpenAI API URL
    },
    
    # Custom OpenAI-compatible Configuration
    "academic-cloud": {
        "enabled": True,
        "name": "academic-cloud",
        "model": "e5-mistral-7b-instruct",
        "context_limit": 4096,
        "embedding_type": "float",  # alternative: base64
        "rate_limit_per_minute": 100,
        "concurrent_requests": 30,
        "base_url": "https://chat-ai.academiccloud.de/v1"
    },
    # X-RateLimit-Limit-Minute: 100
    # X-RateLimit-Limit-Hour: 2000
    # X-RateLimit-Limit-Day: 4000
    # X-RateLimit-Limit-Month: 30000
    # X-RateLimit-Remaining-Minute: 95
    # X-RateLimit-Remaining-Hour: 1995
    # X-RateLimit-Remaining-Day: 3994
    # X-RateLimit-Remaining-Month: 25994
    # RateLimit-Reset: 10
    # RateLimit-Remaining: 95
    # RateLimit-Limit: 100

    # Cohere Configuration 
    "cohere": {
        "enabled": True,
        "name": "cohere",
        "model": "embed-v4.0",
        "dimensions": 1536,
        "context_limit": 128000,
        "input_type": "clustering",  # or "search_document"
        "embedding_types": ["int8"],  # float is the other option
        "rate_limit_per_minute": 2000,
        "concurrent_requests": 20,
        "max_texts_per_call": 96
    }
}

# Select which providers to use (can specify multiple)
active_providers = ["openai", "academic-cloud", "cohere"]
active_providers = [p for p in active_providers if p in providers_config and providers_config[p]["enabled"]]

## API Keys Setup

In [34]:
# Find the .env file and load it
dotenv.load_dotenv(dotenv.find_dotenv())

# Get API keys with fallbacks
api_keys = {
    "openai": os.getenv("OPENAI_API_KEY", ""),
    "academic-cloud": os.getenv("ACADEMIC_CLOUD_API_KEY") or os.getenv("OPENAI_API_KEY", ""),
    "cohere": os.getenv("COHERE_API_KEY", ""),
}

# Check if the required API keys are set
for provider in active_providers:
    if not api_keys[provider]:
        raise ValueError(f"API key for {provider} is not set. Please set {provider.upper()}_API_KEY in your .env file.")


## Utility Functions

In [35]:
# Batched (a function from Python's own itertools cookbook) breaks up a sequence into chunks
def batched(iterable, n):
    """Batch data into tuples of length n. The last batch may be shorter."""
    # batched('ABCDEFG', 3) --> ABC DEF G
    if n < 1:
        raise ValueError('n must be at least one')
    it = iter(iterable)
    while (batch := tuple(itertools.islice(it, n))):
        yield batch

# Tracked semaphore class to limit concurrent requests
class TrackedSemaphore:
    def __init__(self, limit: int):
        self._semaphore = asyncio.Semaphore(limit)
        self._current_tasks = 0
        self._limit = limit

    @property
    def current_tasks(self):
        return self._current_tasks

    @property
    def limit(self):
        return self._limit

    # Context Manager Support
    async def __aenter__(self):
        await self.acquire()
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        self.release()

    # Direct acquire/release support
    async def acquire(self):
        await self._semaphore.acquire()
        self._current_tasks += 1

    def release(self):
        self._current_tasks -= 1
        if self._current_tasks < 0:
            raise ValueError("Semaphore counter went negative. This should not happen.")
        self._semaphore.release()

### Saving functions

In [36]:
def save_dict_pickle(e: dict, path: str) -> None:
    """Save a dictionary to a pickle file."""
    print("Saving dictionary pickle to disk ...")
    with open(path, "wb") as fOut:
        pickle.dump(e, fOut)
    print(f"Saved dictionary with {len(e)} providers to {path}\n")

def save_dict_parquet(e: dict, path: str) -> None:
    """Save a dictionary to a parquet file."""
    print("Saving dictionary parquet to disk ...")
    with open(path, "wb") as fOut:
        pl.DataFrame(e).write_parquet(path)
    print(f"Saved dictionary with {len(e)} providers to {path}\n")

def save_df_pickle(docs: pl.DataFrame, path: str) -> None:
    """Save a dataframe to a pickle file."""
    print("Saving dataframe pickle to disk ...")
    with open(path, "wb") as fOut:
        pickle.dump(docs, fOut)
    print(f"Saved {len(docs)} dataframe entries to {path}\n")

def save_df_parquet(docs: pl.DataFrame, path: str) -> None:
    """Save a dataframe to a parquet file."""
    print("Saving dataframe parquet to disk ...")
    with open(path, "wb") as fOut:
        pl.DataFrame(docs).write_parquet(path)
    print(f"Saved {len(docs)} dataframe entries to {path}\n")

def save_metadata_json(metadata: dict, path: str) -> None:
    """Save processing metadata to a JSON file."""
    print("Saving processing metadata to disk ...")
    with open(path, 'wb') as fOut:
        fOut.write(
            orjson.dumps(
                metadata,
                option=orjson.OPT_INDENT_2 | orjson.OPT_APPEND_NEWLINE
            )
        )
    print(f"Saved processing metadata to {path}\n")

## Provider Class Implementation

In [37]:
class EmbeddingProvider(ABC):
    """Abstract base class for embedding providers."""

    config: Dict[str, Any]
    api_key: str
    delay: float

    @abstractmethod
    async def initialize(self) -> None:
        """Initialize the provider client."""
        pass

    @abstractmethod
    async def create_embedding(self, text: str, semaphore: TrackedSemaphore) -> List[float]:
        """Create an embedding for the given text."""
        pass

    @abstractmethod
    def get_identifier(self) -> str:
        """Return a unique identifier for this provider and model."""
        pass
    
    @abstractmethod
    def get_tokenizer(self, text: str) -> List[int]:
        """Tokenize the input text."""
        pass
    
    @abstractmethod
    def chunk_text(self, text: str) -> List[str]:
        """Chunk text if it exceeds the context window."""
        pass


class OpenAIProvider(EmbeddingProvider):
    """OpenAI embedding provider implementation."""
    
    def __init__(self, config: Dict[str, Any], api_key: str):
        self.config = config
        self.api_key = api_key
        self.client = None
        self.delay = config["concurrent_requests"] * 60.0 / config["rate_limit_per_minute"]

    async def initialize(self) -> None:
        self.client = openai.AsyncOpenAI(
            api_key=self.api_key,
            base_url=self.config["base_url"]
        )

    async def create_embedding(self, text: str, semaphore: TrackedSemaphore) -> List[float]:
        if not self.client:
            await self.initialize()

        async with semaphore:
            # Adjust delay based on the number of parallel requests underway
            current_parallel_requests = semaphore.current_tasks
            adjusted_delay = self.delay * max(1, current_parallel_requests)
            await asyncio.sleep(adjusted_delay)

            # Prepare text chunks - this provider can handle multiple texts in one call
            input_chunks = self.chunk_text(text)

            # Prepare parameters for API requests
            params = {
                "input": text, # input_chunks,
                "model": self.config["model"],
                "encoding_format": self.config["embedding_type"]
            }
            if "dimensions" in self.config:
                params["dimensions"] = self.config["dimensions"]

            # Do the API requests
            retry_count = 3
            for attempt in range(retry_count):
                try:
                    response = await self.client.embeddings.create(**params)
                    embeddings = [response.data[i].embedding for i in range(len(response.data))]

                    # If multiple chunks, average the embeddings
                    if len(embeddings) > 1:
                        print(f"Warning: Received {len(embeddings)} embeddings from OpenAI. Averaging them.")
                        # Calculate weighted average by text length
                        weights = [len(chunk) for chunk in input_chunks]
                        avg_embedding = np.average(embeddings, axis=0, weights=weights)
                        # Normalize
                        avg_embedding = avg_embedding / np.linalg.norm(avg_embedding)
                        return avg_embedding.tolist()
                        # For now, just return the first embedding
                        # return embeddings[0]
                    else:
                        return embeddings[0]

                except Exception as e:
                    if attempt < retry_count - 1:
                        await asyncio.sleep(2 ** attempt)  # Exponential backoff
                    else:
                        raise

    def get_identifier(self) -> str:
        return f"{self.config['name']}_{self.config['model']}"

    def get_tokenizer(self, text: str) -> List[int]:
        # This depends on the model in use, examples are tiktoken or HuggingFace's AutoTokenizer
        if self.config["model"] == "e5-mistral-7b-instruct":
            tokenizer = transformers.AutoTokenizer.from_pretrained('intfloat/e5-mistral-7b-instruct')
            tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=self.config["context_limit"])
            return tokens["input_ids"].squeeze().tolist()
        else:
            tokenizer = tiktoken.encoding_for_model(self.config["model"])
            return tokenizer.encode(text)

    def chunk_text(self, text: str) -> List[str]:
        # This depends on the model in use, examples are tiktoken or HuggingFace's AutoTokenizer
        tokens = []
        if self.config["model"] == "e5-mistral-7b-instruct":
            tokenizer = transformers.AutoTokenizer.from_pretrained('intfloat/e5-mistral-7b-instruct')
            _tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=self.config["context_limit"])
            tokens = _tokens["input_ids"].squeeze().tolist()
        else:
            tokenizer = tiktoken.encoding_for_model(self.config["model"])
            tokens = tokenizer.encode(text)

        chunks = []
        for chunk in batched(tokens, self.config["context_limit"]):
            chunks.append(tokenizer.decode(chunk))

        return chunks


class CohereProvider(EmbeddingProvider):
    """Cohere embedding provider implementation."""
    
    def __init__(self, config: Dict[str, Any], api_key: str):
        self.config = config
        self.api_key = api_key
        self.client = None
        self.delay = config["concurrent_requests"] * 60.0 / config["rate_limit_per_minute"]

    async def initialize(self) -> None:
        # self.client = cohere.AsyncClient(api_key=self.api_key)
        self.client = cohere.ClientV2(api_key=self.api_key)

    async def create_embedding(self, text: str, semaphore: TrackedSemaphore) -> List[float]:
        if not self.client:
            await self.initialize()

        async with semaphore:
            # Adjust delay based on the number of parallel requests underway
            current_parallel_requests = semaphore.current_tasks
            adjusted_delay = self.delay * max(1, current_parallel_requests)
            await asyncio.sleep(adjusted_delay)

            # Prepare text chunks - this provider can handle multiple texts in one call
            input_chunks = self.chunk_text(text)

            # Prepare parameters for API requests
            params = {
                "texts": input_chunks,
                "model": self.config["model"],
                "input_type": self.config["input_type"],
                "embedding_types": self.config["embedding_types"]
            }
            if "dimensions" in self.config:
                params["output_dimension"] = self.config["dimensions"]

            # Do the API requests
            retry_count = 3
            for attempt in range(retry_count):
                try:
                    response = self.client.embed(**params)
                    # The embedding might process multiple numerical data types, but for now we only care about the first one
                    extract_embedding_type = self.config["embedding_types"][0]
                    embeddings = getattr(response.embeddings, extract_embedding_type)

                    # If the response is empty, raise an error
                    if not embeddings:
                        raise ValueError("Received empty embeddings from Cohere.")
                    # If the response is not a list, raise an error
                    if not isinstance(embeddings, list):
                        print(f"Warning: Received non-list embeddings from Cohere. Type: {type(embeddings)}")
                        print(f"Received: {embeddings}")
                        raise ValueError("Received non-list embeddings from Cohere.")
                    # If the response is not a list of lists, raise an error
                    if not all(isinstance(embedding, list) for embedding in embeddings):
                        raise ValueError("Received non-list of lists embeddings from Cohere.")
                    # If the response is not a list of lists of embedding_type numbers, raise an error
                    if extract_embedding_type == "float":
                        if not all(isinstance(embedding, float) for embedding in itertools.chain.from_iterable(embeddings)):
                            raise ValueError("Received non-list of lists of float embeddings from Cohere.")
                    elif extract_embedding_type == "int8":
                        if not all(isinstance(embedding, int) for embedding in itertools.chain.from_iterable(embeddings)):
                            raise ValueError("Received non-list of lists of int8 embeddings from Cohere.")
                    # If the response is not a list of lists of the correct length, print a warning
                    if not all(len(embedding) == self.config["dimensions"] for embedding in embeddings):
                        print(f"Warning: Received embeddings of incorrect length from Cohere. Expected {self.config['dimensions']}, got {len(embeddings[0])}.")
                    
                    # If multiple chunks, average the embeddings
                    if len(embeddings) > 1:
                        # Calculate weighted average by text length
                        weights = [len(chunk) for chunk in input_chunks]
                        avg_embedding = np.average(a=embeddings, axis=0, weights=weights)
                        # Normalize
                        avg_embedding = avg_embedding / np.linalg.norm(avg_embedding)
                        return avg_embedding.tolist()
                    else:
                        return embeddings[0]

                except Exception as e:
                    if attempt < retry_count - 1:
                        await asyncio.sleep(2 ** attempt)  # Exponential backoff
                    else:
                        raise

    def get_identifier(self) -> str:
        return f"{self.config['name']}_{self.config['model']}_{self.config['input_type']}"

    def get_tokenizer(self, text: str) -> List[int]:
        return self.client.tokenize(text=text, model=self.config["model"])

    def chunk_text(self, text: str) -> List[str]:
        # Split text if it exceeds max length
        # Cohere can handle long texts, but has a limit on tokens per call
        tokens = self.client.tokenize(text=text, model=self.config["model"]).tokens
        if len(tokens) <= self.config["context_limit"]:
            return [text]

        # Simplified chunking by character count (Cohere handles larger contexts)
        # We use a rough approximation: avg 4 chars per token
        avg_chars_per_token = 4
        chars_per_chunk = (self.config["context_limit"] * avg_chars_per_token)

        # Split into chunks
        chunks = []
        for i in range(0, len(text), chars_per_chunk):
            chunks.append(text[i:i + chars_per_chunk])

        return chunks


def create_provider(provider_name: str, config: Dict[str, Any], api_key: str) -> EmbeddingProvider:
    """Factory function to create the appropriate provider."""
    if provider_name in ["openai", "academic-cloud"]:
        return OpenAIProvider(config, api_key)
    elif provider_name == "cohere":
        return CohereProvider(config, api_key)
    else:
        raise ValueError(f"Unknown provider: {provider_name}")

## Embedding Generation Functions

In [ ]:
nest_asyncio.apply()

async def retrieve_embeddings(
    provider: EmbeddingProvider,
    semaphore: TrackedSemaphore,
    doc_id: str,
    content: str
) -> Dict[str, Dict[str, List[float]]]:
    """Make an async embedding call for a specific document."""
    provider_name = provider.get_identifier()

    try:
        embeddings = await provider.create_embedding(content, semaphore)
        return {doc_id: {provider_name: embeddings}}

    except Exception as e:
        print(f"Error creating embedding for {doc_id} with provider {provider_name}: {str(e)}")
        return {doc_id: {provider_name : []}}

async def control_embeddings_retrieval(
    provider: EmbeddingProvider,
    semaphore: TrackedSemaphore,
    dfi: pl.DataFrame,
    progress: rich.progress.Progress,
    task_id: rich.progress.TaskID
) -> Dict[str, Dict[str, List[float]]]:
    """Control async embedding calls for a specific provider."""

    # provider_name = provider.get_identifier()
    # task_id = progress.add_task(f"[cyan]{provider_name}", total=len(dfi))
    results = {}

    async def process_row(row):
        res = await retrieve_embeddings(provider, semaphore, row["url"], row["content"])
        progress.advance(task_id)
        return res

    tasks = [asyncio.create_task(process_row(row)) for row in dfi.rows(named=True)]
    completed = await asyncio.gather(*tasks, return_exceptions=True)

    # Combine all results into one dictionary
    for res in completed:
        if not(isinstance(res, BaseException)):
            for doc_id, providers_dict in res.items():
                if doc_id not in results:
                    results[doc_id] = {}
                results[doc_id].update(providers_dict)

    return results

async def get_embeddings_for_provider(
    provider: EmbeddingProvider,
    semaphore: TrackedSemaphore,
    dfi: pl.DataFrame,
    progress: rich.progress.Progress,
    task_id: rich.progress.TaskID
) -> Dict[str, Dict[str, List[float]]]:
    """Get embeddings for a specific provider, using cache when available."""

    provider_name = provider.get_identifier()
    # print(f"Creating embeddings using {provider_name}. This can take a while ...")

    # No embeddings file exists: create embeddings
    if not os.path.exists(file_path_out_embeddings_pickle):
        embeddings = await control_embeddings_retrieval(
            provider, semaphore, dfi, progress, task_id
        )
        return embeddings

    # Embeddings file exists: check if we have embeddings for this provider
    else:
        print(f"Checking cached embeddings for {provider_name}... ", end="")
        with open(file_path_out_embeddings_pickle, "rb") as fIn:
            cache_data = pickle.load(fIn)

        # Check if we have this provider's embeddings
        if provider_name in cache_data:
            print("Found in cache.")
            cached_docs = set(cache_data[provider_name].keys())
            needed_docs = set(dfi["url"])

            # All documents already have embeddings
            if cached_docs.issuperset(needed_docs):
                print(f"Using cached embeddings for {provider_name}.")
                return {doc_id: {provider_name: cache_data[provider_name][doc_id]} 
                        for doc_id in needed_docs}

            # Some documents need to be processed
            else:
                missing_docs = needed_docs - cached_docs
                print(f"Missing {len(missing_docs)} embeddings for {provider_name}.")

                # Process only missing documents
                missing_df = dfi.filter(pl.col("url").is_in(list(missing_docs)))
                new_embeddings = await control_embeddings_retrieval(
                    provider, semaphore, missing_df, progress, task_id
                )

                # Combine cached with new embeddings
                combined = {}
                for doc_id in needed_docs:
                    combined[doc_id] = {}
                    if doc_id in cache_data[provider_name]:
                        combined[doc_id][provider_name] = cache_data[provider_name][doc_id]
                    elif doc_id in new_embeddings:
                        combined[doc_id].update(new_embeddings[doc_id])

                # Update cache
                cache_data[provider_name].update({
                    doc_id: new_embeddings[doc_id][provider_name] 
                    for doc_id in new_embeddings 
                    if provider_name in new_embeddings[doc_id]
                })
                save_dict_pickle(cache_data, file_path_out_embeddings_pickle)

                return combined

        # Provider not in cache
        else:
            print(f"No cached embeddings for {provider_name}.")
            new_embeddings = await control_embeddings_retrieval(
                provider, semaphore, dfi, progress, task_id
            )

            # Add new provider to cache
            cache_data[provider_name] = {
                doc_id: new_embeddings[doc_id][provider_name]
                for doc_id in new_embeddings
                if provider_name in new_embeddings[doc_id]
            }
            save_dict_pickle(cache_data, file_path_out_embeddings_pickle)

            return new_embeddings

async def get_all_embeddings(
    providers: List[EmbeddingProvider],
    dfi: pl.DataFrame
) -> Dict[str, Dict[str, List[float]]]:
    """Get embeddings from all specified providers."""

    # Create a single shared Progress instance
    progress = rich.progress.Progress(
        rich.progress.TextColumn("[bold blue]{task.description}"),
        rich.progress.BarColumn(bar_width=None),
        rich.progress.TimeElapsedColumn(),
        refresh_per_second=2,
        transient=False
    )

    # In get_all_embeddings():
    task_ids = {p.get_identifier(): progress.add_task(p.get_identifier(), total=len(dfi)) for p in providers}

    # Initialize semaphores for each provider
    semaphores = {p.get_identifier(): TrackedSemaphore(p.config["concurrent_requests"]) for p in providers}

    try:
        with progress:
            # Dispatch tasks for all providers using the shared progress instance
            tasks = [
                get_embeddings_for_provider(
                    p,
                    semaphores[p.get_identifier()],
                    dfi,
                    progress,
                    task_ids[p.get_identifier()]
                ) for p in providers
            ]
            # collect results
            provider_results = await asyncio.gather(*tasks, return_exceptions=True)
    finally:
        progress.stop()


    # Combine results from all providers
    all_embeddings = {}
    for r in provider_results:
        if isinstance(r, BaseException):
            print(f"Error occurred while processing provider results: {r}")
        else:
            for doc_id, provider_dict in r.items():
                all_embeddings.setdefault(doc_id, {}).update(provider_dict)

    print("All tasks completed.")
    return all_embeddings

## Load Data

In [39]:
# Load and prepare document data
print(f"Loading documents from {file_path_in_text} ...")
docs = pl.read_csv(file_path_in_text)
print(f"Loaded {len(docs)} documents.")

# Check for required columns
required_columns = ["url", "content"]
for col in required_columns:
    if col not in docs.columns:
        raise ValueError(f"Missing required column: {col}")

# Ensure content is a string
docs = docs.with_columns(pl.col("content").cast(pl.Utf8))

# Filter too short documents
if min_tokens > 0:
    encoding = tiktoken.get_encoding("cl100k_base")
    # Filter by token count
    docs.filter(
        pl.col("content")
        .map_elements(lambda x: len(encoding.encode(x)), return_dtype=pl.Int32)
        >= min_tokens
    )
    print(f"Filtered to {len(docs)} documents with at least {min_tokens} tokens")

# Limit number of documents if specified
if max_documents > 0:
    docs = docs.head(max_documents)
    print(f"Limited to {len(docs)} documents")

# Display basic stats
print(f"Processing {len(docs)} documents")

Loading documents from ./in-data/all_paragraphs_20250411.csv ...
Loaded 147695 documents.
Filtered to 147695 documents with at least 10 tokens
Limited to 5 documents
Processing 5 documents


## Main Execution Block

In [40]:
# Initialize global tracking variables
uploaded_ids = {}

# Initialize providers
providers = [
    create_provider(provider_name, providers_config[provider_name], api_keys[provider_name])
    for provider_name in active_providers
]

# Get embeddings from all configured providers
embeddings_dict = asyncio.run(get_all_embeddings(providers, docs))

Output()

Creating Progress instance
Before progress context


After progress context
All tasks completed.


## Diagnostics, Reporting and Saving

In [41]:
embeddings_dict

{'https://id.salamanca.school/texts/W0103:frontmatter.titlepage': {'openai_text-embedding-3-small': [0.030093893,
   0.017320763,
   0.054670986,
   0.017494716,
   -0.05646022,
   -0.0033113223,
   -0.0098221395,
   0.008169585,
   -0.00069542427,
   -0.0664998,
   0.019060293,
   -0.010760244,
   0.0018731025,
   -0.0054267165,
   0.051987138,
   0.0075296857,
   0.017382888,
   0.035262786,
   0.043637387,
   0.034219068,
   0.008343538,
   0.03941281,
   -0.015233325,
   0.009741376,
   -0.017879898,
   -0.022539357,
   -0.009635761,
   -0.0013263926,
   0.042867023,
   0.03153522,
   0.041798454,
   -0.032081928,
   0.053030856,
   -0.008884035,
   -0.053378765,
   0.043637387,
   0.011219977,
   -0.039686166,
   -0.026813634,
   0.033498403,
   0.008268987,
   -0.060485993,
   0.03046665,
   0.038393945,
   0.011443632,
   0.015804885,
   0.017755646,
   -0.011692136,
   -0.012133231,
   0.02523563,
   0.006697196,
   -0.0063461834,
   -0.0024850448,
   0.062424324,
   -0.0273851

In [42]:
# Merge embeddings to dataframe - create columns for each provider
for provider in providers:
    provider_name = provider.get_identifier()
    column_name = f"embeddings_{provider_name}"

    print(f"Adding {column_name} to dataframe...")
    # Create a mapping function that extracts the right embedding
    def get_embedding(doc_id, p_id=provider_name):
        if doc_id in embeddings_dict and p_id in embeddings_dict[doc_id]:
            return embeddings_dict[doc_id][p_id]
        return None

    docs = docs.with_columns(
        pl.col("url")
        .map_elements(lambda x: get_embedding(x, provider_name), return_dtype=pl.List(pl.Float32))
        .alias(column_name)
    )

Adding embeddings_openai_text-embedding-3-small to dataframe...
Adding embeddings_academic-cloud_e5-mistral-7b-instruct to dataframe...
Adding embeddings_cohere_embed-v4.0_clustering to dataframe...


In [43]:
# Show first few documents and their providers
print("Sample of embeddings:")
sample_doc_id = list(embeddings_dict.keys())[0]
print(f"Document ID: {sample_doc_id}")
print(f"Available embedding providers: {list(embeddings_dict[sample_doc_id].keys())}")

# Display the enriched DataFrame
print("\nFirst rows of enriched docs dataframe:")
docs.head(3)

Sample of embeddings:
Document ID: https://id.salamanca.school/texts/W0103:frontmatter.titlepage
Available embedding providers: ['openai_text-embedding-3-small', 'academic-cloud_e5-mistral-7b-instruct', 'cohere_embed-v4.0_clustering']

First rows of enriched docs dataframe:


url,xmlid,lang,wid,author-id,author-name,title,year,passage,citation-recommendation,content,embeddings_openai_text-embedding-3-small,embeddings_academic-cloud_e5-mistral-7b-instruct,embeddings_cohere_embed-v4.0_clustering
str,str,str,str,str,str,str,i64,str,str,str,list[f32],list[f32],list[f32]
"""https://id.salamanca.school/te…","""W0103-00-0001-tp-03e8""","""la""","""W0103""","""A0089""","""Vacca""","""Expositiones locorum obscurior…",1554,"""""","""Vacca, Expositiones locorum ob…","""ANTONII VACCAE a capite Silici…","[0.030094, 0.017321, … 0.01109]","[0.018145, 0.004694, … 0.005415]","[-51.0, -15.0, … 21.0]"
"""https://id.salamanca.school/te…","""W0103-00-0002-he-03e8""","""la""","""W0103""","""A0089""","""Vacca""","""Expositiones locorum obscurior…",1554,"""priv.""","""Vacca, Expositiones locorum ob…","""BREVIS PRIVILEGII.""","[0.028219, 0.046031, … 0.013539]","[0.014004, -0.00173, … 0.036868]","[-6.0, 19.0, … -11.0]"
"""https://id.salamanca.school/te…","""W0103-00-0002-pa-03eb""","""la""","""W0103""","""A0089""","""Vacca""","""Expositiones locorum obscurior…",1554,"""priv. paragr. 'Ne qvis has loc…","""Vacca, Expositiones locorum ob…","""Ne qvis has locorvm obscvriorv…","[0.022757, 0.024413, … -0.000193]","[0.020142, -0.00378, … 0.012993]","[-52.0, -57.0, … 20.0]"


## Saving Data

In [44]:
# Ensure output directory exists
os.makedirs(file_path_out, exist_ok=True)

# === Save just the embeddings to disk separately ===

# Prepare the cache data format for just the embeddings
cache_data = {}
for provider in providers:
    provider_name = provider.get_identifier()
    cache_data[provider_name] = {
        doc_id: embeddings_dict[doc_id][provider_name]
        for doc_id in embeddings_dict
        if provider_name in embeddings_dict[doc_id]
    }

# - pickle format
print(f"Saving {file_path_out_embeddings_pickle} to disk ...")
save_dict_pickle(cache_data, file_path_out_embeddings_pickle)

# - parquet format
print(f"Saving {file_path_out_embeddings_parquet} to disk ...")
save_dict_parquet(cache_data, file_path_out_embeddings_parquet)


# === Save the configuration to disk ===

processing_metadata = {
    "processing_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "documents": {
        "total_available": len(docs),
        "processed": len(docs) if max_documents == -1 else max_documents,
        "min_tokens_threshold": min_tokens
    },
    "providers": {
        provider.get_identifier(): {
            "name": provider.config["name"],
            "model": provider.config["model"],
            "base_url": provider.config.get("base_url", None),
            "dimensions": provider.config.get("dimensions", None),
            "embedding_type": provider.config.get("embedding_type", None),
            "embedding_types": provider.config.get("embedding_types", None),
            "input_type": provider.config.get("input_type", None),
            "encoding": provider.config.get("encoding", None),
            "context_limit": provider.config.get("context_limit", None),
            "max_texts_per_call": provider.config.get("max_texts_per_call", None),
            "rate_limit_per_minute": provider.config.get("rate_limit_per_minute", None),
            "concurrent_requests": provider.config.get("concurrent_requests", None),
            "documents_processed": len(cache_data.get(provider.get_identifier(), {}))
        }
        for provider in providers
    }
}
save_metadata_json(processing_metadata, file_path_out_config_json)


# === Save the full docs dataframe to disk ===

# - pickle format
print(f"Saving {file_path_out_docs_pickle} to disk ...")
save_df_pickle(docs, file_path_out_docs_pickle)

# - parquet format
print(f"Saving {file_path_out_docs_parquet} to disk ...")
save_df_parquet(docs, file_path_out_docs_parquet)

# - csv format (but without the embeddings, as they are too large for csv)
print(f"Saving {file_path_out_docs_csv} to disk ...")
docs_for_csv = docs.clone().drop(pl.selectors.starts_with("embeddings_"))
docs_for_csv.write_csv(file_path_out_docs_csv)
print("Saving done.")

Saving ./out-data/2025-05-08_embeddings.pkl to disk ...
Saving dictionary pickle to disk ...
Saved dictionary with 3 providers to ./out-data/2025-05-08_embeddings.pkl

Saving ./out-data/2025-05-08_embeddings.parquet to disk ...
Saving dictionary parquet to disk ...
Saved dictionary with 3 providers to ./out-data/2025-05-08_embeddings.parquet

Saving processing metadata to disk ...
Saved processing metadata to ./out-data/2025-05-08_processing_metadata.json

Saving ./out-data/2025-05-08_full_docs.pkl to disk ...
Saving dataframe pickle to disk ...
Saved 5 dataframe entries to ./out-data/2025-05-08_full_docs.pkl

Saving ./out-data/2025-05-08_full_docs.parquet to disk ...
Saving dataframe parquet to disk ...
Saved 5 dataframe entries to ./out-data/2025-05-08_full_docs.parquet

Saving ./out-data/2025-05-08_docs.csv to disk ...
Saving done.


In [45]:
print("\nAll done.\n")


All done.

